In [1]:
import site
import sys
import os
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from scipy.sparse import hstack, kron, eye, csc_matrix, block_diag
import time
import quimb as qu
import quimb.tensor as qtn
from quimb.tensor.fitting import tensor_network_distance
from ncon import ncon
import torch
import warnings
import torch.optim as optim
from quimb.tensor import tensor_split
import cotengra as ctg
from torch.optim.lr_scheduler import ReduceLROnPlateau
folder_path = os.path.abspath('/Users/xiangzou/Desktop/Research_Projects/Quantum Channel Learning') 
sys.path.append(folder_path)
import TDME_Trott as tdme

from autoray import backend_like, compose, dag, do

plt.rcParams["figure.figsize"] = (16, 10)
def get_average_loss_weight_tn(tn, tn_target, N, n_times=50, weight = 1, normalized=False):
    total_loss = 0.0
    for _ in range(n_times):
        M_input, weight = fcl.random_pauli_MPS(N, weight)
        #M_input = ensure_numpy(M_input)
        with torch.no_grad():
            M_realoutput = tn_target | M_input.copy(deep=True)
        M_output = tn | M_input.copy(deep=True)
        loss = tensor_network_distance(M_output.copy().astype("complex64"), M_realoutput.copy().astype("complex64"), normalized=normalized)
        total_loss += loss.item()
        M_outputcopy = M_output.copy()
        for i in range(N):
            M_outputcopy.reindex_({f'input{i}': f'xyz{i}'})
        #print(M_output)
        #print(M_outputcopy)
        #print(M_output.H.astype("complex64") @ M_outputcopy.astype("complex64"))
    average_loss = total_loss / (n_times)
    return average_loss
def gradient_callback(tnopt):
    # Get current vector position
    x = tnopt.vectorizer.vector
    
    # Compute value and gradient
    loss_val, grad = tnopt.vectorized_value_and_grad(x)
    
    # Print gradient statistics
    print(f"\nIteration {tnopt.nevals}:")
    print(f"  Loss: {loss_val:.6f}")
    print(f"  Gradient norm: {np.linalg.norm(grad):.6f}")
    print(f"  Gradient shape: {grad.shape}")
    print(f"  Max gradient component: {np.max(np.abs(grad)):.6f}")
    print(f"  Gradient: {grad}")
    for tensor in tnopt._tn_opt:
        if hasattr(tensor, 'params'):
            print(f"  params shape: {tensor.params.shape}")
            print(f"  requires_grad: {tensor.params.requires_grad}")
            print(f"  initial values: {tensor.params.data}")
            print(f"  gradient: {tensor.params.grad}")

def overlapping_loss_function(tn, tnB):
    return 1 - torch.abs(tn @ tnB)**2

%matplotlib inline
#for 3D non-interactive plots

#set up the size and DPI for the graph
fsize = 12
tsize = 12

tdir = 'in'
major = 7.5
minor = 4.5

style = 'default'

plt.style.use(style)
plt.rcParams['text.usetex'] = False
plt.rcParams['font.size'] = fsize
plt.rcParams['legend.fontsize'] = tsize
plt.rcParams['xtick.direction'] = tdir
plt.rcParams['ytick.direction'] = tdir
plt.rcParams['xtick.major.size'] = major
plt.rcParams['xtick.minor.size'] = minor
plt.rcParams['ytick.major.size'] = major
plt.rcParams['ytick.minor.size'] = minor
plt.rcParams["figure.figsize"] = (16,9)
plt.rcParams['axes.grid']=False
plt.rcParams['grid.alpha'] = 0.25
#mpl.rcParams.update({"axes.grid" : True, "grid.alpha": 0.25})
plt.rcParams['figure.dpi'] = 400
plt.rcParams['text.usetex'] = True

In [2]:
for depolarizing_name in [0, 2, 4, 6, 8, 10]:
    learning_loss_all=[]
    model_all = []
    param_all = []
    p_depolar_all = []
    testing_loss_all = []
    target_layer = [2, 3, 4, 5, 6, 7]
    for t in target_layer:
        depolarizing_strength = depolarizing_name * 0.01
        print(f"depolarizing strength: {depolarizing_strength}, target layer: {t}")
        model, learning_loss, param, p_depolar, testing_loss = tdme.Learning_MPO_scheduler(4, 2, t, depolarizing_strength = depolarizing_strength, epochs=120, lr=0.05, normalized = False , truncation = True,  noise_type = "depolarizing")
        learning_loss_all.append(learning_loss)
        testing_loss_all.append(testing_loss)
        model_all.append(model)
        param_all.append(param)
    np.save(f"Depolarizing_ChangeT_N_L_p_8_2_{depolarizing_name:03d}_learning.npy", np.array(learning_loss_all))
    np.save(f"Depolarizing_ChangeT_N_L_p_8_2_{depolarizing_name:03d}_testing.npy", np.array(testing_loss_all))

depolarizing strength: 0.0, target layer: 2
Processing MPS 1/12
Processing MPS 2/12
Processing MPS 3/12
Processing MPS 4/12
Processing MPS 5/12
Processing MPS 6/12
Processing MPS 7/12
Processing MPS 8/12
Processing MPS 9/12
Processing MPS 10/12
Processing MPS 11/12
Processing MPS 12/12
Epoch 1/120


/home/simba/.virtualenvs/QIP/lib/python3.12/site-packages/cotengra/hyperoptimizers/hyper.py:55: UserWarning: Couldn't find `optuna`, `cmaes`, or `nevergrad` so will use completely random sampling in place of hyper-optimization. It is recommended to install one of these libraries for higher quality contraction paths.
  warnings.warn(


Loss at epoch 0: 1.407631915507188
Learning rate at epoch 0: 0.05
Epoch 2/120
Epoch 3/120
Epoch 4/120
Epoch 5/120
Epoch 6/120
Epoch 7/120
Epoch 8/120
Epoch 9/120
Epoch 10/120


: 